# 在 Google Colab 上訓練 2000 回合

訓練不需要畫面，只用 numpy 和 PyTorch（Colab 都已經內建）。

1. **執行階段 → 變更執行階段類型 → T4 GPU**
2. 把 `vertical-shmup.zip` 上傳到 Google Drive 根目錄（`MyDrive/`）
3. 依序執行每個 cell。權重、log、影片都會存到 `MyDrive/shmup/`，斷線也不會不見。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/shmup'
!mkdir -p $DRIVE

In [ ]:
!rm -rf /content/vertical-shmup
!unzip -q -o /content/drive/MyDrive/vertical-shmup.zip -d /content
%cd /content/vertical-shmup
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, os; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'cpus', os.cpu_count())"

## 訓練 2000 回合
在第 10 / 50 / 500 / 2000 回合各存一份權重到 `MyDrive/shmup/checkpoints/`。log 在 `MyDrive/shmup/train.log`。

In [ ]:
!python -u scripts/train_gpu.py --max-episodes 2000 --save-episodes 10,50,500,2000 --total-steps 100000000 --save-path $DRIVE/dqn.pt 2>&1 | tee -a $DRIVE/train.log

## 斷線後接著訓練
重新執行前兩個 cell，再執行這一個。回合數會從存檔接著算，一樣到 2000 就停。

In [ ]:
!python -u scripts/train_gpu.py --max-episodes 2000 --save-episodes 10,50,500,2000 --total-steps 100000000 --save-path $DRIVE/dqn.pt --resume $DRIVE/dqn_last.pt 2>&1 | tee -a $DRIVE/train.log

## 錄影：第 10 / 50 / 500 / 2000 回合的權重
錄影需要 pygame（只有這一步需要），Colab 本身有 ffmpeg。

In [ ]:
!pip -q install pygame
!python scripts/record_video.py --checkpoint-dir $DRIVE/checkpoints --out $DRIVE/videos
!ls -la $DRIVE/videos

## 匯出網頁版模型並下載

In [ ]:
!python scripts/export_web.py --model $DRIVE/dqn.pt
!cp web/model/dqn_model.js $DRIVE/
from google.colab import files
files.download(DRIVE + '/dqn_model.js')
files.download(DRIVE + '/videos/compare_2x2.mp4')